# 01. Coleta: dengue (SINAN)

Baixa as notificações de dengue do Ministério da Saúde (Brasil, 2000 a 2026) e grava em `data/raw/`.

Há duas formas de coletar. As duas trazem os mesmos dados; muda só o formato e o tempo:

| | Opção A: API | Opção B: CSV anual (usada) |
|---|---|---|
| Como | pedidos à API, 1000 registros por vez | download de um arquivo por ano |
| Tempo | dias | minutos |
| Onde grava | `data/raw/dengue_api/ano=AAAA/dengue.jsonl` | `data/raw/dengue_csv/ano=AAAA/DENGBR{aa}.csv` |

Nenhuma das duas é *scraping*: as duas usam canais oficiais, feitos para programas baixarem dados.

**Neste notebook**
1. Opção A: API
2. Opção B: CSV anual
3. Comparação entre A e B

In [ ]:
import sys, json, subprocess
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
from coleta.dengue import coleta_dengue_api as dengue_api
from coleta.dengue import coleta_dengue_csv as dengue_csv
from config import DATA_RAW, ANO_INICIO, ANO_FIM_COLETA
pd.set_option("display.max_columns", 50)

## 1. Opção A: API Dados Abertos

**Endpoint:** `GET https://apidadosabertos.saude.gov.br/arboviroses/dengue`
(documentação: https://apidadosabertos.saude.gov.br/v1/, seção "Agravo Arboviroses")

| Parâmetro | O que faz | Usamos? |
|---|---|---|
| `nu_ano` | filtra pelo ano | sim, um ano por vez |
| `limit` | registros por página (máximo 1000) | sim, 1000 |
| `offset` | a partir de qual registro começar | sim: 0, 1000, 2000... |
| `id_municip` | filtra pelo município que notificou | não |
| `campos` | escolhe quais campos vêm na resposta | não (queremos todos) |

Exemplo:
```bash
curl 'https://apidadosabertos.saude.gov.br/arboviroses/dengue?nu_ano=2026&limit=1000&offset=0'
```

A resposta é um JSON com a lista de notificações na chave `dengue` (~196 campos cada, todos como texto).
Quando a lista vem vazia, acabaram os dados daquele ano.

**O que testamos na prática** (às vezes diferente da documentação):
- `offset` conta registros, não páginas (a documentação diz "número da página").
- `limit` acima de 1000 não dá erro, mas devolve no máximo 1000.
- Não dá para filtrar por estado: `sg_uf` é ignorado.
- O código do município tem 7 dígitos até 2006 e 6 dígitos de 2007 em diante.
- Páginas muito avançadas (offset alto) ficam lentas, e a API às vezes cai.

### 1.1 Testar a API
Busca 5 registros e mostra o tempo de resposta, os campos e um exemplo.

In [ ]:
amostra = dengue_api.probe(page_size=5)

### 1.2 Testar os filtros
Esperado: 100% para o filtro de ano e perto de 0% para o de estado (que a API ignora).

In [ ]:
dengue_api.testar_filtro(dengue_api.PARAM_ANO, 2023, "nu_ano")
dengue_api.testar_filtro("sg_uf", "24", "sg_uf")   # RN; se der ~100%, a API passou a filtrar UF

### 1.3 Quanto tempo vai levar?
Estimativa a partir do número de registros e do tempo de cada pedido. Ajuste os valores conforme o que medir.

In [ ]:
print(dengue_api.estimar_requisicoes(n_linhas=1_000_000, seg_por_req=3))

### 1.4 Teste rápido
Baixa só 3 páginas de um ano, para ver se tudo funciona. O ano fica "em andamento" e a coleta completa continua dele.

In [ ]:
sessao = dengue_api._sessao()
arq = dengue_api.coletar_particao(sessao, ano=2026, max_paginas=3)
piloto = pd.read_json(arq, lines=True, dtype=False)
print(arq, piloto.shape)
piloto.head()

### 1.5 Coleta completa
Desligada por padrão porque demora muito. Se cair, basta rodar de novo: ela continua de onde parou.
Pelo terminal (dá para dividir os anos entre o grupo):
```bash
python src/coleta/dengue/coleta_dengue_api.py --anos 2000 2026
```

In [ ]:
RODAR_COLETA_COMPLETA = False

if RODAR_COLETA_COMPLETA:
    dengue_api.coletar(range(ANO_INICIO, ANO_FIM_COLETA + 1))

### 1.6 Andamento
Situação de cada ano: completo, em andamento (com o ponto onde parou) ou não iniciado.

In [ ]:
status = []
for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1):
    pasta = DATA_RAW / "dengue_api" / f"ano={ano}"
    final, parcial, estado = pasta / "dengue.jsonl", pasta / "dengue.part", pasta / "dengue.estado.json"
    if final.exists():
        status.append((ano, "completo", None, final.stat().st_size / 1e6))
    elif parcial.exists():
        offset = json.loads(estado.read_text())["offset"] if estado.exists() else None
        status.append((ano, "em andamento", offset, parcial.stat().st_size / 1e6))
    else:
        status.append((ano, "não iniciado", None, 0.0))

status = pd.DataFrame(status, columns=["ano", "situacao", "offset", "tamanho_mb"]).set_index("ano")
print(status["situacao"].value_counts().to_string())
status

## 2. Opção B: CSV anual (OpenDataSUS)

O Ministério da Saúde publica um arquivo compactado por ano no conjunto "Arboviroses - Dengue"
(https://dados.gov.br/dados/conjuntos-dados/arboviroses-dengue). Cada arquivo fica em:
```
https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR{aa}.csv.zip    (aa = 00 a 26)
```
São ~900 MB compactados e 8,5 GB depois de extraídos (2024 é o maior: 1,8 GB).

### 2.1 Coleta
Baixa e extrai o CSV de cada ano. Anos já baixados são pulados. Pelo terminal:
```bash
python src/coleta/dengue/coleta_dengue_csv.py --anos 2000 2026
```

In [ ]:
for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1):
    print(ano, dengue_csv.coletar_ano(ano))

### 2.2 Andamento
Quais anos já foram baixados e o tamanho de cada arquivo.

In [ ]:
baixados = []
for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1):
    csvs = list((dengue_csv.DESTINO / f"ano={ano}").glob("*.csv"))
    baixados.append((ano, csvs[0].name if csvs else None, csvs[0].stat().st_size / 1e6 if csvs else 0.0))
baixados = pd.DataFrame(baixados, columns=["ano", "arquivo", "tamanho_mb"]).set_index("ano")
print("Anos baixados:", baixados["arquivo"].notna().sum(), "de", len(baixados))
baixados

## 3. Comparação entre A e B

### 3.1 Resumo
| | Opção A: API | Opção B: CSV anual |
|---|---|---|
| Como baixa | página de até 1000 registros | um ano inteiro por arquivo |
| Tempo (2000 a 2026) | dias | minutos |
| Arquivo gravado | JSON Lines: um registro por linha | CSV com cabeçalho |
| Nomes das colunas | minúsculas (`id_mn_resi`) | maiúsculas (`ID_MN_RESI`) |
| Colunas por ano | sempre 196 (as que não existem no período vêm vazias) | só as da ficha daquele período (63 a 121) |
| Valor ausente | `"nan"` ou `null` | célula vazia |
| Notificações | as mesmas | as mesmas |

### 3.2 Como os dados chegam
O primeiro registro de 2000 nas duas opções: direto da API (opção A) e no arquivo baixado (opção B).

In [ ]:
ANO_EXEMPLO = 2000
reg_api = dengue_api.buscar_pagina(dengue_api._sessao(), {"limit": 1, "offset": 0, dengue_api.PARAM_ANO: ANO_EXEMPLO})[0]
arq_csv = dengue_csv.DESTINO / f"ano={ANO_EXEMPLO}" / f"DENGBR{ANO_EXEMPLO % 100:02d}.csv"
with open(arq_csv, encoding="latin-1") as f:
    cabecalho_csv, linha_csv = f.readline().strip(), f.readline().strip()

print("Opção A, API: um objeto JSON (trecho)")
print(json.dumps(reg_api, ensure_ascii=False)[:400], "...\n")
print("Opção B, CSV: cabeçalho + uma linha por notificação (trecho)")
print(cabecalho_csv[:300], "...")
print(linha_csv[:300], "...")

CAMPOS = ["nu_ano", "id_municip", "id_mn_resi", "dt_notific", "dt_sin_pri", "cs_sexo",
          "nu_idade", "nu_idade_n", "con_classi", "classi_fin"]
reg_csv = pd.read_csv(arq_csv, dtype=str, nrows=1).iloc[0]
pd.DataFrame({"API (opção A)": pd.Series(reg_api), "CSV (opção B)": pd.Series({k.lower(): v for k, v in reg_csv.items()})}).reindex(CAMPOS)

### 3.3 As duas trazem as mesmas notificações?
Número de registros por ano, nos anos completos nas duas opções. Resultado da comparação feita em 29/10/2026: Pequenas diferenças
são normais: os arquivos foram gerados em datas diferentes.

| Ano | API | CSV | Diferença |
|---|---|---|---|
| 2000 | 173.036 | 172.855 | −181 (−0,10%) |
| 2001 | 488.544 | 488.590 | +46 |
| 2002 | 896.921 | 897.093 | +172 |
| 2003 | 416.598 | 416.609 | +11 |
| 2004 | 136.872 | 136.867 | −5 |
| 2005 | 261.501 | 261.501 | 0 |
| 2006 | 411.022 | 411.022 | 0 |
| 2007 | 717.097 | 717.097 | 0 |
| 2008 | 919.322 | 919.324 | +2 |
| 2009 | 600.658 | 600.658 | 0 |
| 2010 | 1.381.254 | 1.381.254 | 0 |
| 2011 | 1.150.011 | 1.150.011 | 0 |
| 2012 | 952.486 | 950.180 | −2.306 (−0,24%) |

Na mesma comparação, a idade e a classificação dos anos 2000–2006 vieram preenchidas nas duas opções, com os nomes da
ficha antiga (`nu_idade`, `con_classi`) e o mesmo percentual.

A célula abaixo refaz a comparação para os anos que tiverem os arquivos das duas opções em `data/raw/`.


In [ ]:
contagem = []
for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1):
    arq_api = DATA_RAW / "dengue_api" / f"ano={ano}" / "dengue.jsonl"
    arq_csv = dengue_csv.DESTINO / f"ano={ano}" / f"DENGBR{ano % 100:02d}.csv"
    if not (arq_api.exists() and arq_csv.exists()):
        continue
    n_api = int(subprocess.run(["wc", "-l", str(arq_api)], capture_output=True, text=True).stdout.split()[0])
    n_csv = len(pd.read_csv(arq_csv, usecols=["NU_ANO"], dtype=str))
    contagem.append((ano, n_api, n_csv))

contagem = pd.DataFrame(contagem, columns=["ano", "api", "csv"]).set_index("ano")
contagem["diferenca"] = contagem["csv"] - contagem["api"]
print("Anos com as duas opções completas:", len(contagem))
contagem

### 3.4 As colunas mudam com o tempo
A ficha de notificação do SINAN mudou algumas vezes, e o CSV traz só as colunas da ficha de cada período:

| Anos | Colunas | O que muda |
|---|---|---|
| 2000–2006 | 107 | ficha antiga: idade em `NU_IDADE`, classificação em `CON_CLASSI` |
| 2007–2008 | 63 | ficha nova: idade em `NU_IDADE_N`, classificação em `CLASSI_FIN` |
| 2009–2013 | 66 | entra o exame NS1 |
| 2014–2015 | 121 | nova classificação de casos: sinais de alarme, gravidade, doenças pré-existentes |
| 2016–2019 | 119 | data de nascimento no lugar do ano de nascimento |
| 2020–2026 | 121 | volta o ano de nascimento |

As tabelas abaixo mostram, ano a ano, o que entrou e saiu, e em quais anos cada coluna aparece.

In [ ]:
matriz = dengue_csv.comparar_colunas(range(ANO_INICIO, ANO_FIM_COLETA + 1))   # coluna x ano (True = existe)
pd.set_option("display.max_colwidth", 300)
dengue_csv.resumir_mudancas(matriz)

In [ ]:
# colunas que não existem em todos os anos, e em quais anos aparecem
parciais = matriz[~matriz.all(axis=1)]
print(f"{len(matriz)} colunas no total | {matriz.all(axis=1).sum()} em todos os anos | {len(parciais)} só em parte dos anos")
parciais.apply(lambda l: f"{l.index[l].min()}–{l.index[l].max()} ({l.sum()} anos)", axis=1).rename("anos")